# Cross-Encoder Reranking: Two-Stage Retrieval

A **bi-encoder** embeds the query and each document *separately*, which makes retrieval fast but limits accuracy — the model never sees the query and document together. A **cross-encoder** reads the concatenated `(query, document)` pair through one transformer and outputs a single relevance score. It is far more accurate, and far too slow to run against every chunk.

**Two-stage retrieval** gets both properties:

1. A fast base retriever (BM25, dense, or hybrid) fetches a candidate pool (e.g. top-40).
2. The cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) rescores only those candidates, and the top-k by cross-encoder score are returned.

In [1]:
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from rag.data_ingestion import load_documents, chunk_documents
from rag.embeddings import load_cross_encoder
from rag.retrievers import (
    BM25Retriever,
    DenseRetriever,
    HybridRetriever,
    RerankerRetriever,
)

PDF_PATH = ROOT / "data" / "google_10K.pdf"

documents = load_documents(PDF_PATH)
chunks = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)

print(f"Loaded {len(documents)} pages -> {len(chunks)} chunks")

Loaded 107 pages -> 433 chunks


## How a cross-encoder scores

Scores are unbounded logits — only the *ordering* matters, with higher meaning more relevant. Note how sharply it separates the truly relevant sentence from topically-nearby distractors.

In [2]:
cross_encoder = load_cross_encoder()

query = "What were the total revenues?"
candidates = [
    "Total revenues for fiscal 2024 were $402 billion, up 15% from prior year.",
    "Operating expenses increased by 20% compared to 2023.",
    "Annual revenue growth rate over the past 5 years.",
    "The weather was sunny that day.",
]

scores = cross_encoder.predict([[query, doc] for doc in candidates], convert_to_numpy=True)

print(f"Query: {query}\n")
print(f"{'Score':>8}  Document")
print("-" * 70)
for doc, score in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True):
    print(f"{score:>8.3f}  {doc}")

Query: What were the total revenues?

   Score  Document
----------------------------------------------------------------------
   7.412  Total revenues for fiscal 2024 were $402 billion, up 15% from prior year.
  -7.465  Annual revenue growth rate over the past 5 years.
 -10.408  Operating expenses increased by 20% compared to 2023.
 -11.216  The weather was sunny that day.


## Building reranked retrievers

`RerankerRetriever` wraps any base retriever: it pulls `candidate_pool` candidates from the base, rescores them with the cross-encoder, and returns the top-k.

In [3]:
bm25_base = BM25Retriever()
bm25_base.add_documents(chunks)

dense_base = DenseRetriever()
dense_base.add_documents(chunks)

hybrid_base = HybridRetriever()
hybrid_base.add_documents(chunks)

bm25_reranked = RerankerRetriever(base_retriever=bm25_base, candidate_pool=40)
dense_reranked = RerankerRetriever(base_retriever=dense_base, candidate_pool=40)
hybrid_reranked = RerankerRetriever(base_retriever=hybrid_base, candidate_pool=40)

print("Created rerankers wrapping BM25, Dense, and Hybrid retrievers")

Created rerankers wrapping BM25, Dense, and Hybrid retrievers


## Two-stage retrieval, with scores

Stage 1 shows the dense retriever's own top-5; stage 2 shows the top-5 after cross-encoder reranking, with each chunk's cross-encoder score. Reranking often promotes chunks that the bi-encoder ranked lower in the pool.

In [4]:
query = "What were the primary revenue sources?"

stage1 = dense_base.retrieve(query, top_k=40)
reranked = dense_reranked.retrieve(query, top_k=5)
rerank_scores = cross_encoder.predict(
    [[query, doc.page_content] for doc in reranked], convert_to_numpy=True
)

print(f"Query: {query}\n")
print("Stage 1 — dense retrieval (top-5 of the 40-candidate pool):")
for i, doc in enumerate(stage1[:5], start=1):
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  {i}. [page {doc.metadata['page']}] {preview}...")

print("\nStage 2 — after cross-encoder reranking (top-5 with scores):")
for i, (doc, score) in enumerate(zip(reranked, rerank_scores), start=1):
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  {i}. [{score:>7.3f}] [page {doc.metadata['page']}] {preview}...")

Query: What were the primary revenue sources?

Stage 1 — dense retrieval (top-5 of the 40-candidate pool):
  1. [page 43] ments to suppliers for devices, to tax authorities for income taxes, and other g...
  2. [page 65] x benefits of the position recognized in the financial statements are then measu...
  3. [page 59]  the variable interest and voting models. Intercompany balances and transactions...
  4. [page 15]  fraudulently generate revenues, or to otherwise generate traffic that does not ...
  5. [page 66] vices 34,688  40,340  48,030  Google Services total 272,543  304,930  342,721  G...

Stage 2 — after cross-encoder reranking (top-5 with scores):
  1. [ -4.307] [page 59] by our sole ability to monetize the advertising inventory before it is transferr...
  2. [ -4.699] [page 59] d liabilities. Revenue Recognition Revenues are recognized when control of the p...
  3. [ -4.743] [page 43] et cash provided by operating activities $ 125,299  $ 164,713  Net cash used in ...
  4. [ -6

## Does the base retriever matter?

The reranker can only reorder what the base retriever hands it — a relevant chunk missing from the candidate pool is gone for good. Comparing the reranked top-5 across bases shows how much the pool matters.

In [5]:
query = "total revenues by segment"

strategies = {
    "BM25": bm25_base,
    "Dense": dense_base,
    "Hybrid": hybrid_base,
    "BM25+Rerank": bm25_reranked,
    "Dense+Rerank": dense_reranked,
    "Hybrid+Rerank": hybrid_reranked,
}

print(f"Query: {query}\n")
print(f"{'Strategy':<16} Top-5 doc IDs")
print("-" * 75)
for name, r in strategies.items():
    ids = [doc.metadata["doc_id"] for doc in r.retrieve(query, top_k=5)]
    print(f"{name:<16} {ids}")

Query: total revenues by segment

Strategy         Top-5 doc IDs
---------------------------------------------------------------------------
BM25             ['chunk_168', 'chunk_393', 'chunk_294', 'chunk_192', 'chunk_169']
Dense            ['chunk_192', 'chunk_294', 'chunk_396', 'chunk_256', 'chunk_179']
Hybrid           ['chunk_192', 'chunk_294', 'chunk_396', 'chunk_395', 'chunk_179']
BM25+Rerank      ['chunk_396', 'chunk_294', 'chunk_192', 'chunk_184', 'chunk_188']
Dense+Rerank     ['chunk_396', 'chunk_294', 'chunk_192', 'chunk_184', 'chunk_394']
Hybrid+Rerank    ['chunk_396', 'chunk_294', 'chunk_192', 'chunk_184', 'chunk_188']


## Candidate pool size

A larger pool gives the cross-encoder more chances to find the best chunk, but every candidate is one more `(query, document)` pair to score. Here we score the pools directly with the already-loaded cross-encoder to isolate the scoring cost.

In [6]:
import time

query = "financial performance"

print(f"Query: {query}\n")
print(f"{'Pool size':<11} {'Scoring time (ms)':<19} Top-1 after rerank")
print("-" * 55)
for pool in [5, 10, 20, 40, 80]:
    candidates = dense_base.retrieve(query, top_k=pool)
    pairs = [[query, doc.page_content] for doc in candidates]
    start = time.time()
    scores = cross_encoder.predict(pairs, convert_to_numpy=True)
    elapsed = (time.time() - start) * 1000
    top1 = candidates[int(np.argmax(scores))].metadata["doc_id"]
    print(f"{pool:<11} {elapsed:<19.1f} {top1}")

Query: financial performance

Pool size   Scoring time (ms)   Top-1 after rerank
-------------------------------------------------------
5           28.4                chunk_234
10          49.9                chunk_136
20          215.8               chunk_136
40          366.0               chunk_136
80          999.3               chunk_183


## Latency cost of reranking

End-to-end comparison of the dense retriever alone vs. wrapped in a reranker. Note: `RerankerRetriever` loads the cross-encoder on every `retrieve()` call, so the overhead below includes model loading, not just pair scoring — the previous cell shows the scoring cost in isolation.

In [7]:
queries = ["total revenues 2024", "operating income", "cost of goods sold", "stock performance"]

print(f"{'Query':<24} {'Dense (ms)':<12} {'Dense+Rerank (ms)':<18}")
print("-" * 55)
for query in queries:
    start = time.time()
    dense_base.retrieve(query, top_k=5)
    dense_ms = (time.time() - start) * 1000

    start = time.time()
    dense_reranked.retrieve(query, top_k=5)
    rerank_ms = (time.time() - start) * 1000

    print(f"{query:<24} {dense_ms:<12.1f} {rerank_ms:<18.1f}")

Query                    Dense (ms)   Dense+Rerank (ms) 
-------------------------------------------------------
total revenues 2024      5.5          2975.4            
operating income         5.6          3185.2            
cost of goods sold       6.4          2842.8            
stock performance        5.5          3032.8            


## Takeaways

- Cross-encoders are much better relevance judges than bi-encoders, but only affordable on a small candidate pool.
- The reranker can't recover chunks the base retriever never surfaced — pool size and base quality both matter.
- Reranking adds real latency; use it when answer quality justifies the cost.

`05_evaluation.ipynb` measures whether these quality gains actually show up in retrieval metrics.